# Is Shampoo Truly Second-Order or Adam in Rotated Coordinates

**Paper:** [https://arxiv.org/abs/2409.11321](https://arxiv.org/abs/2409.11321)  
**Authors:** Nikhil Vyas, Depen Morwani, Rosie Zhao, Mujin Kwun, Itai Shapira, David Brandfonbrener, Lucas Janson, Sham Kakade  
**Repository:** [https://github.com/nikhilvyas/SOAP](https://github.com/nikhilvyas/SOAP)  
**License:** MIT  

---

*Reproduction generated by Vivory Research — runs on free-tier hardware (Kaggle T4 / Oracle CPU / GitHub Actions).*
*Produced: 2026-05-06 13:14 UTC*


## 1. Setup

Install dependencies from the paper's `requirements.txt`. Some packages may need GPU-specific wheels — adjust for your Colab/Kaggle runtime.

In [ ]:
!pip install --quiet --upgrade pip


## 2. Repository

Clone the reference implementation.

In [ ]:
!git clone --depth 1 https://github.com/nikhilvyas/SOAP
%cd SOAP
!ls -la


## 3. Dataset

Download the dataset. Replace this cell with the dataset-specific loading code from the repository's README or `scripts/download_data.sh`.

In [ ]:
# TODO: Replace with dataset-specific download/load code.
# Check the repo README for instructions — common patterns:
#   bash scripts/download_data.sh
#   python -m src.data.download
#   from datasets import load_dataset; ds = load_dataset("name")
print("Dataset placeholder — fill in from repo README.")


## 4. Configuration

Core hyperparameters. Consider reducing epochs/batch size to fit free-tier GPU limits (Kaggle T4: 16GB VRAM, 30h/week; Colab: variable).

In [ ]:
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Reduced for free-tier — adjust if you have more GPU budget.
CONFIG = {
    "seed": SEED,
    "max_epochs": 1,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "subset_fraction": 0.1,  # use 10% of data for quick reproduction
}
print(json.dumps(CONFIG, indent=2))


## 5+6. Paper-aware evaluation (auto-generated)

The cell below was generated by Vivory's reproduction agent (Opus 4.7) from the paper's abstract, body, repo README, and claimed_metrics. It performs real measurement on a small subset and writes the result to `/kaggle/working/metrics.json` for the runner to ingest.

In [ ]:
!pip install -q transformers datasets

import os, sys, time, json, math, traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    # Locate or clone the SOAP repo
    repo_dir = None
    for d in ["SOAP", "soap"]:
        if os.path.isdir(d) and os.path.isfile(os.path.join(d, "soap.py")):
            repo_dir = d
            break
    if repo_dir is None:
        os.system("git clone -q https://github.com/nikhilvyas/SOAP.git")
        repo_dir = "SOAP"
    sys.path.insert(0, repo_dir)
    from soap import SOAP

    # Minimal Shampoo (1/4 power, EMA stats) for a baseline comparison.
    class SimpleShampoo(torch.optim.Optimizer):
        def __init__(self, params, lr=3e-3, eps=1e-8, update_freq=10, beta=0.95, weight_decay=0.01):
            defaults = dict(lr=lr, eps=eps, update_freq=update_freq, beta=beta, weight_decay=weight_decay)
            super().__init__(params, defaults)

        @torch.no_grad()
        def step(self, closure=None):
            for group in self.param_groups:
                for p in group['params']:
                    if p.grad is None:
                        continue
                    grad = p.grad
                    state = self.state[p]
                    if group['weight_decay'] != 0:
                        p.mul_(1 - group['lr'] * group['weight_decay'])
                    if grad.dim() != 2 or min(grad.shape) < 2:
                        if 'm' not in state:
                            state['m'] = torch.zeros_like(p)
                            state['v'] = torch.zeros_like(p)
                        state['m'].mul_(0.9).add_(grad, alpha=0.1)
                        state['v'].mul_(0.999).addcmul_(grad, grad, value=0.001)
                        p.addcdiv_(state['m'], state['v'].sqrt().add_(group['eps']), value=-group['lr'])
                        continue
                    g = grad.float()
                    m, n = g.shape
                    if 'L' not in state:
                        state['L'] = torch.zeros(m, m, device=p.device)
                        state['R'] = torch.zeros(n, n, device=p.device)
                        state['L_inv4'] = None
                        state['R_inv4'] = None
                        state['step'] = 0
                    state['step'] += 1
                    state['L'].mul_(group['beta']).add_(g @ g.t(), alpha=1 - group['beta'])
                    state['R'].mul_(group['beta']).add_(g.t() @ g, alpha=1 - group['beta'])
                    if state['step'] % group['update_freq'] == 1 or state['L_inv4'] is None:
                        eyeL = group['eps'] * torch.eye(m, device=p.device)
                        eyeR = group['eps'] * torch.eye(n, device=p.device)
                        try:
                            Le, Lv = torch.linalg.eigh(state['L'] + eyeL)
                            Re, Rv = torch.linalg.eigh(state['R'] + eyeR)
                            state['L_inv4'] = (Lv * Le.clamp(min=1e-30).pow(-0.25)) @ Lv.t()
                            state['R_inv4'] = (Rv * Re.clamp(min=1e-30).pow(-0.25)) @ Rv.t()
                        except Exception:
                            pass
                    if state['L_inv4'] is not None:
                        upd = (state['L_inv4'] @ g @ state['R_inv4']).to(p.dtype)
                        p.add_(upd, alpha=-group['lr'])

    # Build a small word-level dataset from wikitext-2 to keep embedding eigendecomp cheap.
    from datasets import load_dataset
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    text = " ".join(t for t in ds["text"] if t.strip())[:1500000]
    from collections import Counter
    words = text.split()
    counter = Counter(words)
    VOCAB = 2000
    top = counter.most_common(VOCAB - 2)
    w2i = {w: i + 2 for i, (w, _) in enumerate(top)}
    ids_full = torch.tensor([w2i.get(w, 1) for w in words], dtype=torch.long)
    print(f"Tokens: {len(ids_full)}, vocab: {VOCAB}")

    SEQ = 128
    BATCH = 16
    N_ITERS = 300

    def get_batch(step):
        g = torch.Generator()
        g.manual_seed(1234 + step)
        idx = torch.randint(0, len(ids_full) - SEQ - 1, (BATCH,), generator=g)
        x = torch.stack([ids_full[i:i + SEQ] for i in idx]).to(device)
        y = torch.stack([ids_full[i + 1:i + SEQ + 1] for i in idx]).to(device)
        return x, y

    class TinyLM(nn.Module):
        def __init__(self, vocab=VOCAB, d=128, nhead=4, nlayers=3, seq_len=SEQ):
            super().__init__()
            self.emb = nn.Embedding(vocab, d)
            self.pos = nn.Embedding(seq_len, d)
            layer = nn.TransformerEncoderLayer(
                d, nhead, dim_feedforward=4 * d, batch_first=True,
                activation='gelu', norm_first=True, dropout=0.0)
            self.tr = nn.TransformerEncoder(layer, nlayers)
            self.norm = nn.LayerNorm(d)
            self.head = nn.Linear(d, vocab, bias=False)

        def forward(self, x):
            t = x.size(1)
            pos = torch.arange(t, device=x.device).unsqueeze(0)
            h = self.emb(x) + self.pos(pos)
            mask = torch.triu(torch.full((t, t), float('-inf'), device=x.device), diagonal=1)
            h = self.tr(h, mask=mask, is_causal=True)
            h = self.norm(h)
            return self.head(h)

    def make_model():
        torch.manual_seed(0)
        torch.cuda.manual_seed_all(0)
        return TinyLM().to(device)

    def run(opt_name):
        model = make_model()
        if opt_name == "adamw":
            opt = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.01)
        elif opt_name == "soap":
            opt = SOAP(model.parameters(), lr=3e-3, betas=(0.95, 0.95),
                       weight_decay=0.01, precondition_frequency=10)
        elif opt_name == "shampoo":
            opt = SimpleShampoo(model.parameters(), lr=3e-3, update_freq=10,
                                beta=0.95, weight_decay=0.01)
        losses, times = [], []
        model.train()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        for step in range(N_ITERS):
            x, y = get_batch(step)
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB), y.reshape(-1))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            losses.append(loss.item())
            times.append(time.time() - t0)
        return np.array(losses), np.array(times)

    print("Training AdamW...")
    aL, aT = run("adamw")
    print(f"  AdamW final-10 mean loss: {aL[-10:].mean():.4f}, total: {aT[-1]:.1f}s")

    print("Training Shampoo...")
    sL, sT = run("shampoo")
    print(f"  Shampoo final-10 mean loss: {sL[-10:].mean():.4f}, total: {sT[-1]:.1f}s")

    print("Training SOAP...")
    pL, pT = run("soap")
    print(f"  SOAP final-10 mean loss: {pL[-10:].mean():.4f}, total: {pT[-1]:.1f}s")

    W = 10
    def smooth(a):
        return np.convolve(a, np.ones(W) / W, mode='valid')

    def iters_to(losses, target):
        sm = smooth(losses)
        idxs = np.where(sm <= target)[0]
        return int(idxs[0]) + W if len(idxs) else len(losses)

    def time_at(times, k):
        return float(times[min(k, len(times) - 1)])

    target_a = float(smooth(aL).min())
    target_s = float(smooth(sL).min())

    ia = iters_to(aL, target_a)
    ip_a = iters_to(pL, target_a)
    is_ = iters_to(sL, target_s)
    ip_s = iters_to(pL, target_s)

    ta = time_at(aT, ia)
    tp_a = time_at(pT, ip_a)
    ts = time_at(sT, is_)
    tp_s = time_at(pT, ip_s)

    def red(b, c):
        return float((b - c) / b * 100.0) if b > 0 else 0.0

    metrics = {
        "iterations_reduction_vs_adamw_percent": red(ia, ip_a),
        "wall_clock_reduction_vs_adamw_percent": red(ta, tp_a),
        "iterations_reduction_vs_shampoo_percent": red(is_, ip_s),
        "wall_clock_reduction_vs_shampoo_percent": red(ts, tp_s),
    }
    print(f"AdamW iters/time to target {target_a:.4f}: {ia} / {ta:.2f}s | SOAP: {ip_a} / {tp_a:.2f}s")
    print(f"Shampoo iters/time to target {target_s:.4f}: {is_} / {ts:.2f}s | SOAP: {ip_s} / {tp_s:.2f}s")

    os.makedirs("/kaggle/working", exist_ok=True)
    with open("/kaggle/working/metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))

except Exception as e:
    traceback.print_exc()
    print("Falling back to documented-limitation sentinel.")
    os.makedirs("/kaggle/working", exist_ok=True)
    with open("/kaggle/working/metrics.json", "w") as f:
        json.dump({"unsupported_infrastructure": 1.0}, f)
    print({"unsupported_infrastructure": 1.0})

## Appendix — Reproduction policy

This notebook runs on **free-tier hardware only**:

- **Kaggle Notebooks** — T4 GPU, 30h/week quota
- **Oracle Cloud** — ARM 4-core CPU, no GPU
- **GitHub Actions** — 2-core CPU, no GPU, 6h timeout
- **Colab** — variable T4/V100, 12h sessions (manual only)

If the full experiment exceeds these limits, reduce `max_epochs` / `subset_fraction` in the config cell and note the delta in the reproduction report.
